# 0.准备各种包

In [ ]:
import pandas as pd

In [ ]:
from collections import Counter
from itertools import product

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import pearsonr
from sklearn.metrics import mean_absolute_error

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import tensorflow as tf
from tensorflow import keras

# 1 两万数据做预测polya

In [ ]:
df = pd.read_csv(
    "/public/Zhaijx/cheny/project/01.polya-evo/03.polya_AI/01.gene_fa/final/all_gene_seq_polya.txt",
    sep="\t",
)

# 查看数据前几行
print(df.head())

In [ ]:
df = df.sample(n=20000, random_state=42)

In [ ]:
print("整张表中每列的 NaN 数：")
print(df.isna().sum())

In [ ]:
from collections import Counter
from itertools import product

# 生成所有可能的 3-mer 组合
k = 3
kmers = ["".join(p) for p in product("ATGC", repeat=k)]


# 计算 3-mer 频率
def kmer_count(seq, k):
    counts = Counter([seq[i : i + k] for i in range(len(seq) - k + 1)])
    total = sum(counts.values())
    return [counts[kmer] / total if total > 0 else 0 for kmer in kmers]


# 应用到所有序列
df_features = pd.DataFrame(
    df["Sequence"].apply(lambda x: kmer_count(x, k)).tolist(),
    columns=kmers,
    index=df.index,  # 关键！保持 index 与原 df 一致
)
df_features["all_median_polya"] = df["all_median_polya"]x

In [ ]:
nan_rows = df_features[df_features["all_median_polya"].isna()]
nan_rows

## 1.2 数据划分

In [ ]:
from sklearn.model_selection import train_test_split

X = df_features.drop(columns=["all_median_polya"])
y = df_features["all_median_polya"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
len(X_test)

## 1.3 先尝试随机森林回归

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# 训练模型
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 预测
y_pred = model.predict(X_test)

# 计算误差
mae = mean_absolute_error(y_test, y_pred)
print(f"Mean Absolute Error: {mae}")

## 1.4 在深度学习（神经网络）

In [ ]:
# 构建神经网络
model = keras.Sequential(
    [
        keras.layers.Dense(128, activation="relu", input_shape=(X_train.shape[1],)),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dense(1),  # 输出层
    ]
)

# 编译模型
model.compile(optimizer="adam", loss="mse", metrics=["mae"])

# 训练模型
history = model.fit(
    X_train, y_train, epochs=50, batch_size=32, validation_data=(X_test, y_test)
)

# 预测
y_pred_nn = model.predict(X_test).flatten()

# 计算误差
mae_nn = mean_absolute_error(y_test, y_pred_nn)
print(f"Neural Network Mean Absolute Error: {mae_nn}")

In [ ]:
mae_nn

## 1.5 比较2种结果

In [ ]:
# 计算 Pearson 相关系数
cor_rf, _ = pearsonr(y_test, y_pred)  # 随机森林相关系数
cor_nn, _ = pearsonr(y_test, y_pred_nn)  # 神经网络相关系数

# 计算 MAE
mae_rf = mean_absolute_error(y_test, y_pred)  # 随机森林 MAE
mae_nn = mean_absolute_error(y_test, y_pred_nn)  # 神经网络 MAE

In [ ]:
import matplotlib.pyplot as plt

# 创建一个横向排列的图像窗口 (1 行, 2 列)
fig, axes = plt.subplots(1, 2, figsize=(6, 3))  # figsize 调整宽度

# 随机森林模型的散点图
axes[0].scatter(y_test, y_pred, alpha=0.5, s=2)
axes[0].set_xlabel("True Poly(A) Length")
axes[0].set_ylabel("Predicted Poly(A) Length")
axes[0].set_title("Random Forest")
# 标注 Pearson 相关系数 & MAE
axes[0].text(0.5, 0.1, f"cor = {cor_rf:.2f}", transform=axes[0].transAxes, fontsize=10)
axes[0].text(0.5, 0.2, f"MAE = {mae_rf:.2f}", transform=axes[0].transAxes, fontsize=10)

# 神经网络模型的散点图
axes[1].scatter(y_test, y_pred_nn, alpha=0.5, s=2, color="red")
axes[1].set_xlabel("True Poly(A) Length")
axes[1].set_ylabel("Predicted Poly(A) Length")
axes[1].set_title("Neural Network")
# 标注 Pearson 相关系数 & MAE
axes[1].text(0.5, 0.1, f"cor = {cor_nn:.2f}", transform=axes[1].transAxes, fontsize=10)
axes[1].text(0.5, 0.2, f"MAE = {mae_nn:.2f}", transform=axes[1].transAxes, fontsize=10)

# 调整布局，防止标题重叠
plt.tight_layout()

# 显示图像
plt.show()

## 1.6 kmer 6

In [ ]:
from collections import Counter
from itertools import product

# 生成所有可能的 6-mer 组合
k = 6
kmers = ["".join(p) for p in product("ATGC", repeat=k)]


# 计算 6MER
def kmer_count(seq, k):
    counts = Counter([seq[i : i + k] for i in range(len(seq) - k + 1)])
    total = sum(counts.values())
    return [counts[kmer] / total if total > 0 else 0 for kmer in kmers]


# 应用到所有序列
df_features = pd.DataFrame(
    df["Sequence"].apply(lambda x: kmer_count(x, k)).tolist(),
    columns=kmers,
    index=df.index,  # <—— 保证行号一致
)
df_features["all_median_polya"] = df["all_median_polya"]

print(df_features.head())

In [ ]:
X = df_features.drop(columns=["all_median_polya"])
y = df_features["all_median_polya"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
## 随机训练模型
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=4)
model.fit(X_train, y_train)
# 预测
y_pred = model.predict(X_test)
# 计算误差
mae = mean_absolute_error(y_test, y_pred)
print(f"Mean Absolute Error: {mae}")

In [ ]:
# 神经网络
model = keras.Sequential(
    [
        keras.layers.Dense(128, activation="relu", input_shape=(X_train.shape[1],)),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dense(1),  # 输出层
    ]
)

# 编译模型
model.compile(optimizer="adam", loss="mse", metrics=["mae"])

# 训练模型
history = model.fit(
    X_train, y_train, epochs=50, batch_size=32, validation_data=(X_test, y_test)
)

# 预测
y_pred_nn = model.predict(X_test).flatten()

# 计算误差
mae_nn = mean_absolute_error(y_test, y_pred_nn)
print(f"Neural Network Mean Absolute Error: {mae_nn}")

In [ ]:
# 计算 Pearson 相关系数
cor_rf, _ = pearsonr(y_test, y_pred)  # 随机森林相关系数
cor_nn, _ = pearsonr(y_test, y_pred_nn)  # 神经网络相关系数

# 计算 MAE
mae_rf = mean_absolute_error(y_test, y_pred)  # 随机森林 MAE
mae_nn = mean_absolute_error(y_test, y_pred_nn)  # 神经网络 MAE
import matplotlib.pyplot as plt

# 创建一个横向排列的图像窗口 (1 行, 2 列)
fig, axes = plt.subplots(1, 2, figsize=(6, 3))  # figsize 调整宽度

# 随机森林模型的散点图
axes[0].scatter(y_test, y_pred, alpha=0.5, s=2)
axes[0].set_xlim(0, 200)
axes[0].set_ylim(0, 200)
axes[0].set_xlabel("True Poly(A) Length")
axes[0].set_ylabel("Predicted Poly(A) Length")
axes[0].set_title("Random Forest")
# 标注 Pearson 相关系数 & MAE
axes[0].text(0.5, 0.1, f"cor = {cor_rf:.2f}", transform=axes[0].transAxes, fontsize=10)
axes[0].text(0.5, 0.2, f"MAE = {mae_rf:.2f}", transform=axes[0].transAxes, fontsize=10)

# 神经网络模型的散点图
axes[1].scatter(y_test, y_pred_nn, alpha=0.5, s=2, color="red")
axes[1].set_xlim(0, 200)
axes[1].set_ylim(0, 200)
axes[1].set_xlabel("True Poly(A) Length")
axes[1].set_ylabel("Predicted Poly(A) Length")
axes[1].set_title("Neural Network")
# 标注 Pearson 相关系数 & MAE
axes[1].text(0.5, 0.1, f"cor = {cor_nn:.2f}", transform=axes[1].transAxes, fontsize=10)
axes[1].text(0.5, 0.2, f"MAE = {mae_nn:.2f}", transform=axes[1].transAxes, fontsize=10)

# 调整布局，防止标题重叠
plt.tight_layout()

# 显示图像
plt.show()

In [ ]:
# 计算 Pearson 相关系数
cor_rf, _ = pearsonr(y_test, y_pred)  # 随机森林相关系数
cor_nn, _ = pearsonr(y_test, y_pred_nn)  # 神经网络相关系数

# 计算 MAE
mae_rf = mean_absolute_error(y_test, y_pred)  # 随机森林 MAE
mae_nn = mean_absolute_error(y_test, y_pred_nn)  # 神经网络 MAE
import matplotlib.pyplot as plt

# 创建一个横向排列的图像窗口 (1 行, 2 列)
fig, axes = plt.subplots(1, 2, figsize=(6, 3))  # figsize 调整宽度

# 随机森林模型的散点图
axes[0].scatter(y_test, y_pred, alpha=0.5, s=2)
axes[0].set_xlim(0, 200)
axes[0].set_ylim(0, 120)
axes[0].set_xlabel("True Poly(A) Length")
axes[0].set_ylabel("Predicted Poly(A) Length")
axes[0].set_title("Random Forest")
# 标注 Pearson 相关系数 & MAE
axes[0].text(0.5, 0.1, f"cor = {cor_rf:.2f}", transform=axes[0].transAxes, fontsize=10)
axes[0].text(0.5, 0.2, f"MAE = {mae_rf:.2f}", transform=axes[0].transAxes, fontsize=10)

# 神经网络模型的散点图
axes[1].scatter(y_test, y_pred_nn, alpha=0.5, s=2, color="red")
axes[1].set_xlim(0, 200)
axes[1].set_ylim(0, 200)
axes[1].set_xlabel("True Poly(A) Length")
axes[1].set_ylabel("Predicted Poly(A) Length")
axes[1].set_title("Neural Network")
# 标注 Pearson 相关系数 & MAE
axes[1].text(0.5, 0.1, f"cor = {cor_nn:.2f}", transform=axes[1].transAxes, fontsize=10)
axes[1].text(0.5, 0.2, f"MAE = {mae_nn:.2f}", transform=axes[1].transAxes, fontsize=10)

# 调整布局，防止标题重叠
plt.tight_layout()

# 显示图像
plt.show()

In [ ]:
15023398 - 15000000

## 1.7  CNN

In [ ]:
df.head()

In [ ]:
seq_lengths = df["Sequence"].apply(len)
print(seq_lengths.describe())
plt.figure(figsize=(10, 6))
plt.hist(seq_lengths, bins=200)
plt.xlabel("Sequence Length")
plt.ylabel("Count")
plt.title("Sequence Length Distribution")
plt.xlim(0, 50000)
plt.show()

In [ ]:
max_len = int(seq_lengths.quantile(0.9))  # 覆盖90%的序列
print(f"选择序列长度: {max_len} bp")

# 或者直接使用最大值
# max_len = seq_lengths.max()

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

# 步骤1：分析序列长度分布
seq_lengths = df["Sequence"].apply(len)
print("序列长度统计:")
print(seq_lengths.describe())

# 步骤2：确定合理的max_len（示例取95%分位数）
max_len = int(seq_lengths.quantile(0.95))
print(f"\n选择max_len为: {max_len} (覆盖95%的序列)")


# 步骤3：智能截断序列（保留末端300bp + 开头200bp）
def process_sequence(seq, max_len):
    """处理超长序列的智能截断策略"""
    seq_len = len(seq)
    if seq_len <= max_len:
        return seq.ljust(max_len, "N")
    else:
        # 保留末端300bp（可能包含polyA信号）和开头200bp
        head = seq[:200]
        tail = seq[-300:]
        combined = head + tail
        return combined.ljust(max_len, "N")[:max_len]


# 步骤4：应用处理
processed_seqs = [process_sequence(s, max_len) for s in df["Sequence"]]

# 步骤5：验证处理结果
sample_idx = np.random.randint(0, len(df), 5)
for idx in sample_idx:
    orig = df["Sequence"].iloc[idx]
    proc = processed_seqs[idx]
    print(f"\n样本{idx}原始长度: {len(orig)}")
    print(f"处理后长度: {len(proc)}")
    print(f"末端示例: ...{proc[-50:]}")


# 步骤6：One-hot编码优化（处理长序列）
def chunked_one_hot(seq, chunk_size=2000):
    """分块处理长序列的编码"""
    encoded = []
    for i in range(0, len(seq), chunk_size):
        chunk = seq[i : i + chunk_size]
        encoded_chunk = one_hot_encode(chunk)
        encoded.append(encoded_chunk)
    return np.concatenate(encoded)


def one_hot_encode(seq):
    """优化后的编码函数"""
    mapping = {
        "A": [1, 0, 0, 0],
        "T": [0, 1, 0, 0],
        "C": [0, 0, 1, 0],
        "G": [0, 0, 0, 1],
        "N": [0, 0, 0, 0],
    }
    return np.array([mapping.get(c.upper(), [0, 0, 0, 0]) for c in seq])


# 步骤7：并行编码（使用joblib加速）
from joblib import Parallel, delayed

X_encoded = Parallel(n_jobs=-1, prefer="threads")(
    delayed(chunked_one_hot)(seq) for seq in processed_seqs
)
X_final = np.array(X_encoded)

# 步骤8：数据分割
y = df["all_median_polya"].values
X_train, X_test, y_train, y_test = train_test_split(
    X_final,
    y,
    test_size=0.2,
    random_state=42,
    stratify=np.digitize(y, bins=np.linspace(y.min(), y.max(), 5)),
)

print(f"\n最终数据维度: {X_final.shape}")
print(f"训练集: {X_train.shape}, 测试集: {X_test.shape}")

# 2 20个物种的数据

In [ ]:
df = pd.read_csv(
    "/public/Zhaijx/cheny/project/01.polya-evo/03.polya_AI/01.gene_fa/final/all_gene_seq_polya.txt",
    sep="\t",
)

In [ ]:
df = df.dropna(subset=["Sequence", "all_median_polya"])

In [ ]:
print(df["Sequence"].apply(type).value_counts())

In [ ]:
bad_rows

In [ ]:
bad_rows = df[~df["Sequence"].apply(lambda x: isinstance(x, str))]
print(bad_rows)

In [ ]:
from collections import Counter
from itertools import product

# 生成所有可能的 3-mer 组合
k = 3
kmers = ["".join(p) for p in product("ATGC", repeat=k)]


# 计算 3-mer 频率
def kmer_count(seq, k):
    counts = Counter([seq[i : i + k] for i in range(len(seq) - k + 1)])
    total = sum(counts.values())
    return [counts[kmer] / total if total > 0 else 0 for kmer in kmers]


# 应用到所有序列
df_features = pd.DataFrame(
    df["Sequence"].apply(lambda x: kmer_count(x, k)).tolist(), columns=kmers
)
df_features["all_median_polya"] = df["all_median_polya"]
print(df_features.head())

In [ ]:
# 分50%出来训练
from sklearn.model_selection import train_test_split

X = df_features.drop(columns=["all_median_polya"])
y = df_features["all_median_polya"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.5, random_state=42
)

In [ ]:
# 创建一个布尔掩码，标记 y_train 中非 NaN 的位置
mask = ~y_train.isna()

# 用这个掩码过滤 X_train 和 y_train
X_train = X_train[mask]
y_train = y_train[mask]
mask = (~y_test.isna()) & (~np.isnan(y_pred))

# 应用掩码清理数据
y_test = y_test[mask]
y_pred = y_pred[mask]

In [ ]:
print("y_test 中 NaN 数:", y_test.isna().sum())
print("y_pred 中 NaN 数:", np.isnan(y_pred).sum())

## 2.1 3-k-mer

In [ ]:
# 随机森林模型
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 预测
y_pred = model.predict(X_test)

# 计算误差
mae = mean_absolute_error(y_test, y_pred)
print(f"Mean Absolute Error: {mae}")

In [ ]:
# 神经网络模型
model = keras.Sequential(
    [
        keras.layers.Dense(128, activation="relu", input_shape=(X_train.shape[1],)),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dense(1),  # 输出层
    ]
)

# 编译模型
model.compile(optimizer="adam", loss="mse", metrics=["mae"])

# 训练模型
history = model.fit(
    X_train, y_train, epochs=50, batch_size=32, validation_data=(X_test, y_test)
)

# 预测
y_pred_nn = model.predict(X_test).flatten()

# 计算误差
mae_nn = mean_absolute_error(y_test, y_pred_nn)
print(f"Neural Network Mean Absolute Error: {mae_nn}")

In [ ]:
# 比较2种数据看看
cor_rf, _ = pearsonr(y_test, y_pred)  # 随机森林相关系数
cor_nn, _ = pearsonr(y_test, y_pred_nn)  # 神经网络相关系数
mae_rf = mean_absolute_error(y_test, y_pred)  # 随机森林 MAE
mae_nn = mean_absolute_error(y_test, y_pred_nn)  # 神经网络 MAE

In [ ]:
import matplotlib.pyplot as plt

# 创建一个横向排列的图像窗口 (1 行, 2 列)
fig, axes = plt.subplots(1, 2, figsize=(6, 3))  # figsize 调整宽度

# 随机森林模型的散点图
axes[0].scatter(y_test, y_pred, alpha=0.5, s=2)
axes[0].set_xlabel("True Poly(A) Length")
axes[0].set_ylabel("Predicted Poly(A) Length")
axes[0].set_title("Random Forest")
# 标注 Pearson 相关系数 & MAE
axes[0].text(0.5, 0.1, f"cor = {cor_rf:.2f}", transform=axes[0].transAxes, fontsize=10)
axes[0].text(0.5, 0.2, f"MAE = {mae_rf:.2f}", transform=axes[0].transAxes, fontsize=10)

# 神经网络模型的散点图
axes[1].scatter(y_test, y_pred_nn, alpha=0.5, s=2, color="red")
axes[1].set_xlabel("True Poly(A) Length")
axes[1].set_ylabel("Predicted Poly(A) Length")
axes[1].set_title("Neural Network")
# 标注 Pearson 相关系数 & MAE
axes[1].text(0.5, 0.1, f"cor = {cor_nn:.2f}", transform=axes[1].transAxes, fontsize=10)
axes[1].text(0.5, 0.2, f"MAE = {mae_nn:.2f}", transform=axes[1].transAxes, fontsize=10)

# 调整布局，防止标题重叠
plt.tight_layout()

# 显示图像
plt.show()